# Exploratory Data Analysis: 2015 Nepal Earthquake Crisis Tweets

**Author:** Yashodeep Basnet, Pratik Gyawali, Safal Pandey  
**Course / Project:** Advance ML Final Assignment — Topic Modeling of Humanitarian Needs  
**Dataset:** `QCRI/CrisisBench-all-lang` (Humanitarian Subset, English Filtered)

---

## 1. Overview and Objectives
This notebook explores social media text posted during the April 2015 Nepal Earthquake.
We investigate:
1. **Volume & Class Balance:** Distribution of human-annotated humanitarian categories (`class_label`).
2. **Text Characteristics:** Tweet length, token distributions, vocabulary diversity.
3. **Data Quality & Noise:** Identification of retweets, URLs, mentions, duplicates, and non-informative text.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load dataset
data_path = Path("../data/interim/nepal_eq.csv")
if not data_path.exists():
    data_path = Path("data/interim/nepal_eq.csv")

df = pd.read_csv(data_path)
print(f"Total Filtered Tweets: {len(df):,}")
df.head()

### Initial Data Inspection & Schema
The dataset contains filtered crisis communications with schema columns including `id`, `event`, `source`, `text`, `lang`, and `class_label`.

In [ ]:
# Humanitarian Category Distribution
class_counts = df["class_label"].value_counts()

plt.figure(figsize=(12, 6))
ax = sns.barplot(x=class_counts.values, y=class_counts.index, palette="viridis")
plt.title("Distribution of Ground-Truth Humanitarian Categories (2015 Nepal Earthquake)", fontsize=14, fontweight="bold")
plt.xlabel("Tweet Count", fontsize=12)
plt.ylabel("Humanitarian Category", fontsize=12)

for i, v in enumerate(class_counts.values):
    ax.text(v + 15, i, f"{v:,} ({v/len(df):.1%})", va="center", fontsize=10)

plt.tight_layout()
plt.show()

### Analysis of Humanitarian Categories
- The distribution shows a high prevalence of sympathy/support and other useful informational posts alongside critical tactical categories: `injured_or_dead_people`, `infrastructure_and_utilities_damage`, `requests_or_urgent_needs`, and `rescue_volunteering_or_donation_effort`.
- This class breakdown will serve as our external validation benchmark for unsupervised LDA topic discovery.

In [ ]:
# Text Length Distribution Analysis
df["char_length"] = df["text"].apply(len)
df["word_count"] = df["text"].apply(lambda t: len(t.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["char_length"], bins=40, ax=axes[0], color="#2b5c8f", kde=True)
axes[0].set_title("Character Length Distribution", fontweight="bold")
axes[0].set_xlabel("Character Count")

sns.histplot(df["word_count"], bins=30, ax=axes[1], color="#d95f02", kde=True)
axes[1].set_title("Word Count Distribution", fontweight="bold")
axes[1].set_xlabel("Word Count")

plt.tight_layout()
plt.show()

print(df[["char_length", "word_count"]].describe())

### Text Length & Granularity Findings
- Tweets have an average length of ~120 characters and ~18 words, reflecting Twitter's historical 140-character limit.
- Because documents are short and sparse, domain stopword filtering and bigram/trigram collocation modeling (`gensim.models.Phrases`) are essential to form cohesive topic representations.

In [ ]:
# Data Origin Sub-Datasets
source_counts = df["source"].value_counts()
plt.figure(figsize=(10, 4))
sns.barplot(x=source_counts.values, y=source_counts.index, palette="magma")
plt.title("Originating Sub-Datasets in CrisisBench", fontweight="bold")
plt.xlabel("Count")
plt.show()

## 2. Key Takeaways for Topic Modeling
1. **Sparsity Mitigation:** Short tweets require rigorous lemmatization and n-gram extraction so multi-word crisis concepts (e.g., `medical_supplies`, `search_rescue`, `death_toll`) are preserved.
2. **Stopword Strategy:** Standard stopwords must be supplemented with high-frequency event tokens (`nepal`, `earthquake`, `kathmandu`, `rt`, `http`) to prevent trivial topics dominated by the disaster name itself.
3. **Candidate Validation:** Unsupervised discovered topics will be mapped back to these annotated classes using semantic embedding cosine similarity to quantify alignment.